In [12]:
import requests
import zipfile
import io
import pandas as pd
import plotly

# Download the zip file for January 2018
url = "https://s3.amazonaws.com/tripdata/JC-201710-citibike-tripdata.csv.zip"
response = requests.get (url)

# Extract the CSV file from the zip
with zipfile.ZipFile (io.BytesIO (response.content)) as z:
    # List files in the ZIP to find the CSV
    csv_file = [f for f in z.namelist () if f.endswith ('.csv')] [0]
    with z.open (csv_file) as f:
        df = pd.read_csv (f)

# Display the first 5 rows (by default)
df.head ()

# Check the dataset's columns and data types
df.info ()

<class 'pandas.DataFrame'>
RangeIndex: 34919 entries, 0 to 34918
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   tripduration             34919 non-null  int64  
 1   starttime                34919 non-null  str    
 2   stoptime                 34919 non-null  str    
 3   start station id         34919 non-null  int64  
 4   start station name       34919 non-null  str    
 5   start station latitude   34919 non-null  float64
 6   start station longitude  34919 non-null  float64
 7   end station id           34919 non-null  int64  
 8   end station name         34919 non-null  str    
 9   end station latitude     34919 non-null  float64
 10  end station longitude    34919 non-null  float64
 11  bikeid                   34919 non-null  int64  
 12  usertype                 34919 non-null  str    
 13  birth year               32761 non-null  float64
 14  gender                   34919 no

In [20]:
len_pre_clean = len (df)

# Convert timestamps to datetime
df ['starttime'] = pd.to_datetime (df ['starttime'])
df ['stoptime'] = pd.to_datetime (df ['stoptime'])

# Calculate the trip duration in minutes
df ['tripduration'] = (df ['stoptime'] - df ['starttime']).dt.total_seconds () / 60

# Remove trips with unrealistic destinations (i.e., less than 1 minute OR greater than 24 hours)
df = df [(df ['tripduration'] >= 1) & (df ['tripduration'] <= 1440)]

# Handle missing station names by filling in with 'Unknown'
df ['start station name'].fillna ("Unknown", inplace = True)
df ['end station name'].fillna ("Unknown", inplace = True)

# Drop rows with missing coordinates
df = df.dropna (subset = ['start station latitude', 'start station longitude', 'end station latitude', 'end station longitude'])

# Display cleaned data summary
df.info ()

len_post_clean = len (df)
deleted_files = len_pre_clean - len_post_clean

print (f"Original length of dataframe is {len_pre_clean} ")
print (f"Length of sanitized data frame: {len_post_clean}")
print (f"{(deleted_files / len_pre_clean) * 100}% of row content was deleted. ")

<class 'pandas.DataFrame'>
Index: 34913 entries, 0 to 34918
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   tripduration             34913 non-null  float64       
 1   starttime                34913 non-null  datetime64[us]
 2   stoptime                 34913 non-null  datetime64[us]
 3   start station id         34913 non-null  int64         
 4   start station name       34913 non-null  str           
 5   start station latitude   34913 non-null  float64       
 6   start station longitude  34913 non-null  float64       
 7   end station id           34913 non-null  int64         
 8   end station name         34913 non-null  str           
 9   end station latitude     34913 non-null  float64       
 10  end station longitude    34913 non-null  float64       
 11  bikeid                   34913 non-null  int64         
 12  usertype                 34913 non-null  str    

/tmp/ipykernel_9272/4147302451.py:14: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df ['start station name'].fillna ("Unknown", inplace = True)
/tmp/ipykernel_9272/4147302451.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method 

In [26]:
# Extract hour from start time
df ['hour'] = df ['starttime'].dt.hour

# Aggregates rides by hour 
hourly_rides = df.groupby ('hour').size ().reset_index (name = 'ride_count')

# Aggregates rides by start station
station_rides = df.groupby('start station name').agg(
    start_latitude=('start station latitude', 'first'),
    start_longitude=('start station longitude', 'first'),
    ride_count=('bikeid', 'count')
).reset_index()

# Top 10 stations by ride count
top_stations = station_rides.nlargest (10, 'ride_count')

# Aggregate rides by day of the week
df ['day_of_week'] = df ['starttime'].dt.day_name ()
day_rides = df.groupby ('day_of_week').size ().reset_index (name = 'ride_count')

# Aggregate by member type
member_rides = df.groupby ('usertype').size ().reset_index (name = 'ride_count')

# Display sample aggregated data
hourly_rides.head ()


,hour,ride_count
0,0,305
1,1,171
2,2,87
3,3,63
4,4,104


In [31]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create a subplot layout (2 rows, 3 columns)
fig = make_subplots (
    rows = 2, cols = 3,
    specs = [[{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
       [{"type": "domain"}, {"type": "xy"}, {"type": "scattergeo"}]],
    
    subplot_titles = ("Rides by Hour", "Top 10 Stations", "Trip Duration Distribution",
                    "Rides by Member Type", "Rides by Day", "Station Map")
)

#1 Line Chart: Rides by Hour
fig.add_trace (
    go.Scatter (x = hourly_rides ['hour'], y = hourly_rides ['ride_count'], mode = 'lines+markers',   
                    name = 'Rides'),
    row = 1, col = 1
)

#2 Bar Chart: Top 10 Stations
fig.add_trace (
    go.Bar (x = top_stations ['start station name'], y = top_stations ['ride_count'], name = 'Stations'),
    row = 1, col = 2
)

#3 Box Plot: Trip Duration Distribution
fig.add_trace (
    go.Box (y = df ['tripduration'], name = 'Duration', boxpoints = 'outliers'),
    row = 1, col = 3
)

#4 Pie Chart: Rides by Member Type
fig.add_trace (
    go.Pie (labels = member_rides ['usertype'], values = member_rides ['ride_count'], name = 'Member Type'),
    row = 2, col = 1
)

#5 Bar Chart: Rides by Day of the Week 
fig.add_trace (
    go.Bar (x = day_rides ['day_of_week'], y = day_rides ['ride_count'], name = 'Day'),
    row = 2, col = 2
)

#6 Geoplot: Station Map
fig.add_trace (
    go.Scattergeo (
        lon = station_rides ['start_longitude'],
        lat = station_rides ['start_latitude'],
        text = station_rides ['start station name'],
        mode = 'markers',
        marker = dict (
            size = station_rides ['ride_count'] / station_rides ['ride_count'].max () * 20,
            color = station_rides ['ride_count'],
            colorscale = "Viridis",
            showscale = True
        ),
        name = 'Stations'
    ),
    row =2, col = 3
)

# Update geoplot layout
fig.update_geos (
    scope = 'usa',
    projection_type = 'mercator',
    center = dict (lat = 40.7128, lon = -74.0060), # NYC Coordinates
    lataxis_range=[40.5, 41.0],
    lonaxis_range=[-74.3, -73.7]
)

# Update overall layout
fig.update_layout (
    title_text = 'NYC Citi Bike Trip Data Dashboard',
    showlegend = False,
    height = 800
)

# Show the dashboard
fig.show ()

In [16]:
# contains the absolute path to the executable code 
import sys

print (sys.executable)

/workspaces/iteractive-plotly-dashboard/.venv/bin/python
